In [3]:
# pip install pyvisa pyvisa-py numpy matplotlib
# On lab PCs with NI-VISA installed, pyvisa will usually use NI-VISA automatically.

import pyvisa
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import os

In [5]:
print(os.getcwd())

C:\Users\ece-houck-j409\PycharmProjects\HouckLab_QICK\WorkingProjects\QM_Team\qubit_measurements\Client_modules\Experiments


In [28]:
sa_addr = "GPIB1::18::INSTR"
save_folder = "V:/t1Team/Data/2026-3-9_BFC_Cooldown/CSTQ02/SpectrumAnalyzer/"
os.makedirs(save_folder, exist_ok=True)

In [26]:
rm = pyvisa.ResourceManager()
sa = rm.open_resource(sa_addr)
print(sa)
sa.close()

GPIBInstrument at GPIB1::18::INSTR


In [29]:
import pyvisa, time

rm = pyvisa.ResourceManager()
print(rm.list_resources())

sa = rm.open_resource("GPIB1::18::INSTR")
sa.timeout = 1000
sa.write_termination = "\n"
sa.read_termination = "\n"
sa.send_end = True

# Try clearing the interface/device
try:
    sa.clear()
except Exception as e:
    print("clear failed:", e)

# Write/read separately so we know where it fails
try:
    print("writing...")
    sa.write("*IDN?")
    time.sleep(0.2)
    print("reading...")
    print(sa.read())
except Exception as e:
    print("IDN failed:", e)

# Try a more old-instrument-friendly termination
try:
    sa.write_termination = "\r\n"
    sa.read_termination = "\n"
    sa.clear()
    print(sa.query("*IDN?"))
except Exception as e:
    print("CRLF failed:", e)

('GPIB0::12::INSTR', 'GPIB0::14::INSTR', 'GPIB0::16::INSTR', 'GPIB0::1::INSTR', 'GPIB0::6::INSTR', 'GPIB0::9::INSTR', 'GPIB1::12::INSTR', 'GPIB1::18::INSTR', 'GPIB1::9::INSTR')
clear failed: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.
writing...
IDN failed: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.
CRLF failed: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.


In [12]:
center = 2.3063e9
span = 2e6
rbw = 100
vbw = 100
points = 401
ref_level = -40

sa.timeout = 30000  # ms

print(sa.query("*IDN?"))

# Basic spectrum analyzer setup
sa.write("*CLS")
sa.write(":INIT:CONT OFF")                 # single sweep mode
sa.write(f":SENS:FREQ:CENT {center}")
sa.write(f":SENS:FREQ:SPAN {span}")
sa.write(f":SENS:BAND:RES {rbw}")
sa.write(f":SENS:BAND:VID {vbw}")
sa.write(f":SENS:SWE:POIN {points}")
sa.write(f":DISP:WIND:TRAC:Y:RLEV {ref_level}")
sa.write(":FORM:TRAC:DATA ASC")           # easy text transfer; slower but robust

# Trigger one sweep and wait until complete
sa.write(":INIT:IMM")
sa.query("*OPC?")

# Read trace
raw = sa.query(":TRAC:DATA? TRACE1")
amp_dbm = np.array([float(x) for x in raw.strip().split(",")])

freq = np.linspace(center - span/2, center + span/2, len(amp_dbm))

tstamp = datetime.now().strftime("%Y%m%d_%H%M%S")
base = save_folder / f"SA_trace_{center/1e9:.6f}GHz_{tstamp}"

np.savez(base.with_suffix(".npz"), freq=freq, amp_dbm=amp_dbm)

plt.figure()
plt.plot(freq / 1e9, amp_dbm)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Power (dBm)")
plt.grid(True)
plt.title("Spectrum Analyzer Trace")
plt.savefig(base.with_suffix(".png"), dpi=200)
plt.show()

sa.close()

Keysight Technologies,N5241B,MY53211681,A.17.30.08



KeyboardInterrupt: 